## Логистическая регрессия

Данные о сессиях пригодятся для решения задачи бинарной классификации — предсказания, завершится ли сессия покупкой.
Вот перечень колонок датасета:
- `Administrative` — количество посещённых административных страниц (например, страницы информации о сайте).
- `Administrative_Duration` — суммарное время (в секундах), проведённое на административных страницах.
- `Informational` — количество посещённых информационных страниц (например, страницы с описанием товаров).
- `Informational_Duration` — суммарное время на информационных страницах.
- `ProductRelated` — количество посещённых продуктовых страниц (страницы с товарами).
- `ProductRelated_Duration` — суммарное время на продуктовых страницах.
- `BounceRates` — доля пользователей, покинувших сайт сразу после просмотра одной страницы.
- `ExitRates` — доля пользователей, покинувших сайт с определённой страницы.
- `PageValues` — значение страницы (оценка важности страницы для совершения покупки).
- `SpecialDay` — близость сессии к особым датам (например, праздникам, когда бывают скидки).
- `Weekend` — был ли визит совершён в выходной день.
- `Revenue` — целевая переменная: совершил ли пользователь покупку в ходе этой сессии. Это бинарный признак, который мы будем предсказывать с помощью модели.

In [11]:
import pandas as pd

# Загрузка данных
df = pd.read_csv('../data/online_shoppers_intention_prepared.csv')

# Посмотрим на форму датасета
print(f"Размер выборки: {df.shape}\n")

# Проверим типы данных
print("Типы данных")
print(df.dtypes)

# Проверим наличие пропусков
print("\nКоличество пропусков:")
print(df.isnull().sum())

Размер выборки: (12330, 12)

Типы данных
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Weekend                      int64
Revenue                      int64
dtype: object

Количество пропусков:
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Weekend                    0
Revenue                    0
dtype: int64


## Разделение выборки

Разделим данные на три части: обучающую ( train ), валидационную ( valid ) и тестовую ( test ) в соотношении 60:20:20.

Стратификация — это способ деления выборки так, чтобы доли классов в train и test были одинаковыми. 
Для логистической регрессии нужно делать shuffle, а для линейной stratify

In [12]:
from sklearn.model_selection import train_test_split

# Разделим признаки и целевую переменную
X = df.drop(columns = ['Revenue'])
y = df['Revenue']

# Разделите данные с учётом стратификации по целевой переменной.
# Cначала выделите 60% на обучение и 40% для валидации и теста (поделим их далее).
X_train, X_temp, y_train, y_temp = train_test_split(X, y, stratify=y, test_size=0.4,  random_state=627)

# Разделите оставшиеся данные (X_temp, y_temp) поровну, чтобы получилось по 20% на валидацию и тест.
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, stratify=y_temp, test_size=0.5, random_state=627)

print("Размер обучающей выборки:", X_train.shape)
print("Размер валидационной выборки:", X_valid.shape)

Размер обучающей выборки: (7398, 11)
Размер валидационной выборки: (2466, 11)


## Шаг 3. Масштабирование признаков

In [13]:
# Вычисляем среднее и стандартное отклонение на обучающей выборке
means = X_train.mean()
stds = X_train.std()

# Масштабируем обучающую выборку
X_train_scaled = (X_train - means) / stds

# Масштабируем валидационную выборку
X_valid_scaled = (X_valid - means) / stds 

## Обучение базовой модели логистической регрессии


In [14]:
# Добавьте импорт класса LogisticRegression из sklearn.linear_model
from sklearn.linear_model import LogisticRegression

# Создаём объект модели логистической регрессии с параметрами по умолчанию
model = LogisticRegression()

# Вызовите метод fit, передав ему 2 параметра:
# X_train — признаковые столбцы, y_train — целевая переменная
model.fit(X_train, y_train)

# Получаем вероятностные предсказания на тестовой выборке
# Для каждого объекта получаем вероятность принадлежности к классу 1 (покупка)
y_pred_proba_valid = model.predict_proba(X_valid)[:, 1]
y_pred_proba_train = model.predict_proba(X_train)[:, 1]

# Посмотрим на первые 5 вероятностей
print(y_pred_proba_valid[:10])

[0.17643923 0.06993007 0.04425886 0.04570515 0.42223462 0.07138517
 0.62095398 0.09531024 0.04734044 0.10815668]


c:\Projects\yandex-practicum\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


#### Интерпретация

Из 10 клиентов модель предсказывает покупку только для одного: значение 0.62 больше 0.5, тогда как остальные вероятности меньше 0.5.

## Метод predict

Иногда от модели нам нужны не вероятности, а конкретное заключение: купит пользователь что-то или нет. В этом случае нужно получить не число от 0 до 1, а просто 0 или 1 — то есть метку класса.

Чтобы получить такие предсказания, используется метод predict. Он сам преобразует вероятности в метки классов по простому правилу:

In [15]:
# Получаем метки классов на тестовой выборке
y_pred_valid = model.predict(X_valid)

# Посмотрим на первые 10 предсказанных меток
print(y_pred_valid[:10])

[0 0 0 0 0 0 1 0 0 0]


## Оценка качества через log loss


In [16]:
from sklearn.metrics import log_loss

# Посчитаем log loss для обучающих и валидационных данных.
loss_train = log_loss(y_train, y_pred_proba_train)
loss_valid = log_loss(y_valid, y_pred_proba_valid)

print(f'Train log loss: {loss_train:.6f}' )
print(f'Valid log loss: {loss_valid:.6f}' )

Train log loss: 0.316081
Valid log loss: 0.314528


Чем меньше значение log loss, тем увереннее и точнее модель предсказывает вероятности:
- если log loss близок к нулю, модель почти не ошибается;
- если log loss большой, модель часто сильно ошибается в вероятностях.

В нашем случае значения получились не очень большими, причём для  валидационных данных предсказания оказались немного лучше, чем для обучающих.

### Ограничение вероятности


In [17]:
import numpy as np

def manual_log_loss(y_true, y_pred_proba):
    eps = 1e-15
    # Ограничиваем вероятности значениями от eps до 1 - eps, 
    # чтобы избежать логарифма от нуля или единицы
    y_pred_proba = np.clip(y_pred_proba, eps, 1 - eps)
    
    # Считаем log loss для каждого объекта по формуле:
    # -(y_true * log(p) + (1 - y_true) * log(1 - p))
    log_loss_elements = -(y_true * np.log(y_pred_proba) + (1 - y_true) * np.log(1 - y_pred_proba))
    
    # Вычисляем среднее значение по всей выборке
    return np.mean(log_loss_elements)

# Проверка
loss = manual_log_loss(np.array([1, 0, 1, 0]), np.array([0.9, 0.2, 0.8, 0.1]))
print("Log loss:", loss)


Log loss: 0.16425203348601802


Сравним библиотечную и нашу log_loss

In [18]:
import numpy as np
from sklearn.metrics import log_loss

def manual_log_loss(y_true, y_pred_proba):
	eps = 1e-15
	y_pred_proba = np.clip(y_pred_proba, eps, 1 - eps)
	loss = -np.mean(y_true * np.log(y_pred_proba) + (1 - y_true) * np.log(1 - y_pred_proba))
	return loss

y_valid = np.array([1, 0, 1, 0])
y_pred_proba_valid = np.array([0.9, 0.2, 0.8, 0.1])

# log loss из библиотеки.
loss_valid_sklearn = log_loss(y_valid, y_pred_proba_valid)

# log loss, написанный вручную.
loss_valid_manual = manual_log_loss(y_valid, y_pred_proba_valid)

print(f'Log loss (sklearn): {loss_valid_sklearn:.6f}')
print(f'Log loss (ручной расчёт): {loss_valid_manual:.6f}')

Log loss (sklearn): 0.164252
Log loss (ручной расчёт): 0.164252
